# Ultimate Tic-Tac-Toe -- self-play training (GPU)

Writes out the training code, then runs batched self-play + training. Make sure **Settings -> Accelerator -> GPU** is enabled before running.


In [ ]:
import torch
print('cuda available:', torch.cuda.is_available())


In [ ]:
%%writefile game.py
"""Ultimate Tic-Tac-Toe game engine.

Board layout: 9 sub-boards of 3x3, indexed 0-8 (row-major, big board).
Each sub-board has 9 cells, indexed 0-8 (row-major).
Cell value: 0 = empty, 1 = player 1 (X), -1 = player 2 (O).

State representation for the NN:
  - 81 cells (current player's marks = +1, opponent's marks = -1, empty = 0)
  - 9 sub-board results (0 = undecided, +1 = current player won it,
    -1 = opponent won it, 2 = drawn/full) -> encoded as one-hot(4) each = 36
  - 9 legal-sub-board flags (which big cell you're allowed to play in) = 9
Total input length = 81 + 36 + 9 = 126
"""
import numpy as np

WIN_LINES = [
    (0, 1, 2), (3, 4, 5), (6, 7, 8),
    (0, 3, 6), (1, 4, 7), (2, 5, 8),
    (0, 4, 8), (2, 4, 6),
]


def check_winner(cells9):
    """cells9: length-9 array of -1/0/1. Returns 1, -1, 0 (no winner)."""
    for a, b, c in WIN_LINES:
        s = cells9[a] + cells9[b] + cells9[c]
        if s == 3:
            return 1
        if s == -3:
            return -1
    return 0


class UltimateTTT:
    def __init__(self):
        self.board = np.zeros((9, 9), dtype=np.int8)
        self.sub_result = np.zeros(9, dtype=np.int8)  # 0 undecided, 1/-1 winner, 2 drawn
        self.player = 1
        self.active_sub = -1  # -1 means "any non-finished sub-board"
        self.winner = 0  # 0 ongoing, 1/-1 winner, 2 draw
        self.done = False

    def clone(self):
        g = UltimateTTT.__new__(UltimateTTT)
        g.board = self.board.copy()
        g.sub_result = self.sub_result.copy()
        g.player = self.player
        g.active_sub = self.active_sub
        g.winner = self.winner
        g.done = self.done
        return g

    def legal_subboards(self):
        if self.active_sub != -1 and self.sub_result[self.active_sub] == 0:
            return [self.active_sub]
        return [s for s in range(9) if self.sub_result[s] == 0]

    def legal_moves(self):
        """Returns list of (sub, cell) legal moves."""
        moves = []
        for s in self.legal_subboards():
            for c in range(9):
                if self.board[s, c] == 0:
                    moves.append((s, c))
        return moves

    def play(self, sub, cell):
        assert not self.done
        assert self.sub_result[sub] == 0
        assert self.board[sub, cell] == 0
        subs_allowed = self.legal_subboards()
        assert sub in subs_allowed

        self.board[sub, cell] = self.player

        w = check_winner(self.board[sub])
        if w != 0:
            self.sub_result[sub] = w
        elif np.all(self.board[sub] != 0):
            self.sub_result[sub] = 2  # drawn sub-board

        big_w = check_winner(self.sub_result.astype(np.int8))
        # treat drawn sub-boards (2) as neutral for big-board win check
        big_cells = np.where(self.sub_result == 2, 0, self.sub_result)
        big_w = check_winner(big_cells)
        if big_w != 0:
            self.winner = big_w
            self.done = True
        elif np.all(self.sub_result != 0):
            self.winner = 2
            self.done = True
        else:
            # next active sub-board = `cell` index, unless that board is finished
            self.active_sub = cell if self.sub_result[cell] == 0 else -1

        self.player *= -1
        return self.done

    def encode(self):
        """Encode state from perspective of the player about to move."""
        p = self.player
        cells = (self.board.flatten() * p).astype(np.float32)  # 81

        sub_onehot = np.zeros((9, 4), dtype=np.float32)
        for s in range(9):
            r = self.sub_result[s]
            if r == 0:
                sub_onehot[s, 0] = 1
            elif r == p:
                sub_onehot[s, 1] = 1
            elif r == -p:
                sub_onehot[s, 2] = 1
            else:
                sub_onehot[s, 3] = 1
        sub_onehot = sub_onehot.flatten()  # 36

        legal_sub_flag = np.zeros(9, dtype=np.float32)
        for s in self.legal_subboards():
            legal_sub_flag[s] = 1

        return np.concatenate([cells, sub_onehot, legal_sub_flag])  # 126

    def legal_action_mask(self):
        """81-length 0/1 mask over (sub*9+cell) actions."""
        mask = np.zeros(81, dtype=np.float32)
        for s, c in self.legal_moves():
            mask[s * 9 + c] = 1
        return mask

    def result_for(self, player):
        """Terminal value from `player`'s perspective: 1 win, -1 loss, 0 draw."""
        if self.winner == 2:
            return 0.0
        return 1.0 if self.winner == player else -1.0


In [ ]:
%%writefile mcts.py
"""Simple PUCT-style MCTS guided by a policy+value function (AlphaZero-lite).

The policy/value function is a callable: fn(state) -> (policy[81], value[float]).
Works with either a random/uniform prior (for bootstrap self-play) or a
trained NN.
"""
import math
import numpy as np


class Node:
    __slots__ = ("game", "parent", "prior", "children", "visit_count",
                 "value_sum", "expanded")

    def __init__(self, game, parent=None, prior=0.0):
        self.game = game
        self.parent = parent
        self.prior = prior
        self.children = {}  # action -> Node
        self.visit_count = 0
        self.value_sum = 0.0
        self.expanded = False

    def value(self):
        if self.visit_count == 0:
            return 0.0
        return self.value_sum / self.visit_count


C_PUCT = 1.5


def puct_score(parent, child, action_prior):
    q = -child.value()  # from parent's perspective (opponent moved)
    u = C_PUCT * action_prior * math.sqrt(parent.visit_count) / (1 + child.visit_count)
    return q + u


def run_mcts(root_game, policy_value_fn, n_simulations=100, dirichlet_eps=0.25):
    root = Node(root_game.clone())
    expand(root, policy_value_fn, add_noise=dirichlet_eps > 0, eps=dirichlet_eps)

    for _ in range(n_simulations):
        node = root
        path = [node]

        while node.expanded and not node.game.done:
            action, node = select_child(node)
            path.append(node)

        if node.game.done:
            leaf_value = node.game.result_for(node.game.player * -1)
            # result_for is from perspective of `player`; node.game.player already
            # flipped after last move, so the mover's perspective is -player
        else:
            leaf_value = expand(node, policy_value_fn, add_noise=False)

        backpropagate(path, leaf_value)

    return root


def select_child(node):
    total_prior_actions = node.children.items()
    best_score = -1e9
    best_action = None
    best_child = None
    for action, child in total_prior_actions:
        score = puct_score(node, child, child.prior)
        if score > best_score:
            best_score = score
            best_action = action
            best_child = child
    return best_action, best_child


def build_children(node, policy, value, add_noise=False, eps=0.25):
    """Shared expansion logic: given a raw policy/value for `node` (already
    computed, single or batched), mask+renormalize and create child nodes.
    """
    game = node.game
    if game.done:
        node.expanded = True
        return game.result_for(game.player * -1)

    mask = game.legal_action_mask()
    policy = policy * mask
    s = policy.sum()
    if s > 1e-8:
        policy = policy / s
    else:
        policy = mask / mask.sum()

    if add_noise:
        legal_idx = np.where(mask > 0)[0]
        noise = np.random.dirichlet([0.3] * len(legal_idx))
        for i, idx in enumerate(legal_idx):
            policy[idx] = 0.75 * policy[idx] + 0.25 * noise[i]

    for sub, cell in game.legal_moves():
        a = sub * 9 + cell
        child_game = game.clone()
        child_game.play(sub, cell)
        node.children[a] = Node(child_game, parent=node, prior=policy[a])

    node.expanded = True
    return value


def expand(node, policy_value_fn, add_noise=False, eps=0.25):
    game = node.game
    if game.done:
        node.expanded = True
        return game.result_for(game.player * -1)
    policy, value = policy_value_fn(game)
    return build_children(node, policy, value, add_noise=add_noise, eps=eps)


def backpropagate(path, leaf_value):
    # leaf_value is from the perspective of the player about to move at the leaf.
    # Alternate sign as we go up (each ancestor is the opponent's turn).
    v = leaf_value
    for node in reversed(path):
        node.visit_count += 1
        node.value_sum += v
        v = -v


def run_mcts_batch(root_games, policy_value_fn_batch, n_simulations=100,
                    dirichlet_eps=0.25):
    """Runs MCTS on many independent games in lockstep, batching every NN
    evaluation across all of them (one forward call per 'wave' instead of
    one per game) -- this is what actually lets a GPU help, since a single
    tiny MLP call on a batch of 1 leaves the GPU almost idle.

    policy_value_fn_batch: fn(states: np.ndarray[N, 126]) -> (policy[N,81], value[N])
    Returns: list of root Nodes, one per input game.
    """
    roots = [Node(g.clone()) for g in root_games]

    root_states = np.stack([r.game.encode() for r in roots])
    root_policies, root_values = policy_value_fn_batch(root_states)
    for r, p, v in zip(roots, root_policies, root_values):
        build_children(r, p, v, add_noise=dirichlet_eps > 0, eps=dirichlet_eps)

    for _ in range(n_simulations):
        leaves = []  # (root_idx, path, leaf_node)
        for i, root in enumerate(roots):
            node = root
            path = [node]
            while node.expanded and not node.game.done:
                _, node = select_child(node)
                path.append(node)
            leaves.append((i, path, node))

        pending_idx = [k for k, (_, _, node) in enumerate(leaves) if not node.game.done]
        if pending_idx:
            states = np.stack([leaves[k][2].game.encode() for k in pending_idx])
            policies, values = policy_value_fn_batch(states)

        ptr = 0
        for k, (i, path, node) in enumerate(leaves):
            if node.game.done:
                leaf_value = node.game.result_for(node.game.player * -1)
            else:
                leaf_value = build_children(node, policies[ptr], values[ptr])
                ptr += 1
            backpropagate(path, leaf_value)

    return roots


def action_probs(root, temperature=1.0):
    counts = np.zeros(81, dtype=np.float32)
    for action, child in root.children.items():
        counts[action] = child.visit_count
    if temperature == 0:
        probs = np.zeros(81, dtype=np.float32)
        probs[np.argmax(counts)] = 1.0
        return probs
    counts = counts ** (1.0 / temperature)
    total = counts.sum()
    if total <= 0:
        mask = root.game.legal_action_mask()
        return mask / mask.sum()
    return counts / total


In [ ]:
%%writefile net_torch.py
"""PyTorch version of the tiny policy+value MLP, for GPU-accelerated
self-play + training on Kaggle. Same architecture as net.py, and saves
checkpoints in the same flat-array .npz format so export_weights.py
(-> C header) works unchanged.
"""
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

INPUT_DIM = 126
H1 = 96
H2 = 64
POLICY_DIM = 81


class PolicyValueNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(INPUT_DIM, H1)
        self.fc2 = nn.Linear(H1, H2)
        self.policy_head = nn.Linear(H2, POLICY_DIM)
        self.value_head = nn.Linear(H2, 1)

    def forward(self, x):
        a1 = F.relu(self.fc1(x))
        a2 = F.relu(self.fc2(a1))
        logits = self.policy_head(a2)
        policy = F.softmax(logits, dim=-1)
        value = torch.tanh(self.value_head(a2))
        return policy, value.squeeze(-1)


def make_batch_fn(model, device):
    """Returns fn_batch(states: np.ndarray[N,126]) -> (policy[N,81], value[N])
    for use by mcts.run_mcts_batch. Runs under no_grad on `device`.
    """
    model.eval()

    @torch.no_grad()
    def fn_batch(states):
        x = torch.from_numpy(states.astype(np.float32)).to(device)
        policy, value = model(x)
        return policy.cpu().numpy(), value.cpu().numpy()

    return fn_batch


def save_npz(model, path):
    sd = model.state_dict()
    params = {
        "W1": sd["fc1.weight"].detach().cpu().numpy().T.astype(np.float32),
        "b1": sd["fc1.bias"].detach().cpu().numpy().astype(np.float32),
        "W2": sd["fc2.weight"].detach().cpu().numpy().T.astype(np.float32),
        "b2": sd["fc2.bias"].detach().cpu().numpy().astype(np.float32),
        "Wp": sd["policy_head.weight"].detach().cpu().numpy().T.astype(np.float32),
        "bp": sd["policy_head.bias"].detach().cpu().numpy().astype(np.float32),
        "Wv": sd["value_head.weight"].detach().cpu().numpy().T.astype(np.float32),
        "bv": sd["value_head.bias"].detach().cpu().numpy().astype(np.float32),
    }
    np.savez(path, **params)


def load_npz_into(model, path):
    data = np.load(path)
    sd = model.state_dict()
    sd["fc1.weight"].copy_(torch.from_numpy(data["W1"].T))
    sd["fc1.bias"].copy_(torch.from_numpy(data["b1"]))
    sd["fc2.weight"].copy_(torch.from_numpy(data["W2"].T))
    sd["fc2.bias"].copy_(torch.from_numpy(data["b2"]))
    sd["policy_head.weight"].copy_(torch.from_numpy(data["Wp"].T))
    sd["policy_head.bias"].copy_(torch.from_numpy(data["bp"]))
    sd["value_head.weight"].copy_(torch.from_numpy(data["Wv"].T))
    sd["value_head.bias"].copy_(torch.from_numpy(data["bv"]))
    return model


In [ ]:
%%writefile train_kaggle.py
"""GPU-accelerated self-play + training for Kaggle notebooks.

Runs many self-play games in lockstep (`--parallel`) so every MCTS leaf
evaluation is batched into a single NN forward call -- this is what actually
uses the GPU; running one game at a time (like train.py) leaves it idle.

Usage (inside a Kaggle notebook cell, GPU accelerator enabled):
    !python train_kaggle.py --iterations 100 --parallel 128 --sims 150 \
        --out /kaggle/working/policy_params.npz

Checkpoints (.npz, same format as train.py) are saved after every iteration,
so you can stop the notebook / hit the free-GPU quota and still keep the
latest weights. Download policy_params.npz and run export_weights.py
locally (or in the same notebook) to produce the C header.
"""
import argparse
import time
import numpy as np
import torch

from game import UltimateTTT
from mcts import run_mcts_batch, action_probs
import net_torch


def self_play_batch(fn_batch, n_parallel, sims, temp_moves, rng):
    games = [UltimateTTT() for _ in range(n_parallel)]
    trajectories = [[] for _ in range(n_parallel)]
    move_no = [0] * n_parallel
    finished = []  # (states, policies, values, winner)
    active = list(range(n_parallel))

    while active:
        active_games = [games[i] for i in active]
        roots = run_mcts_batch(active_games, fn_batch, n_simulations=sims,
                                dirichlet_eps=0.25)
        next_active = []
        for pos, i in enumerate(active):
            root = roots[pos]
            temperature = 1.0 if move_no[i] < temp_moves else 0.0
            pi = action_probs(root, temperature=temperature)
            trajectories[i].append((games[i].encode(), pi, games[i].player))

            if temperature > 0:
                action = int(rng.choice(81, p=pi))
            else:
                action = int(np.argmax(pi))
            sub, cell = divmod(action, 9)
            games[i].play(sub, cell)
            move_no[i] += 1

            if games[i].done:
                states = [s for s, _, _ in trajectories[i]]
                policies = [p for _, p, _ in trajectories[i]]
                players = [pl for _, _, pl in trajectories[i]]
                z = [games[i].result_for(pl) for pl in players]
                finished.append((states, policies, z, games[i].winner))
            else:
                next_active.append(i)
        active = next_active

    return finished


def train_step(model, optimizer, device, X, P, Z, batch_size, epochs, l2=1e-4):
    n = X.shape[0]
    rng = np.random.default_rng()
    losses = []
    Xt_full = torch.from_numpy(X).to(device)
    Pt_full = torch.from_numpy(P).to(device)
    Zt_full = torch.from_numpy(Z).to(device)

    model.train()
    for _ in range(epochs):
        idx = rng.permutation(n)
        for start in range(0, n, batch_size):
            bidx = idx[start:start + batch_size]
            b = torch.from_numpy(bidx).to(device)
            xb = Xt_full[b]
            pb = Pt_full[b]
            zb = Zt_full[b]

            policy, value = model(xb)
            eps = 1e-8
            policy_loss = -(pb * torch.log(policy + eps)).sum(dim=1).mean()
            value_loss = torch.mean((zb - value) ** 2)
            l2_loss = sum((p ** 2).sum() for p in model.parameters()) * l2
            loss = policy_loss + value_loss + l2_loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            losses.append((loss.item(), policy_loss.item(), value_loss.item()))

    return np.mean(losses, axis=0) if losses else (0.0, 0.0, 0.0)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--iterations", type=int, default=100)
    ap.add_argument("--parallel", type=int, default=128,
                     help="self-play games run simultaneously per iteration")
    ap.add_argument("--sims", type=int, default=150)
    ap.add_argument("--temp-moves", type=int, default=10)
    ap.add_argument("--batch-size", type=int, default=512)
    ap.add_argument("--epochs", type=int, default=2)
    ap.add_argument("--lr", type=float, default=1e-3)
    ap.add_argument("--max-replay", type=int, default=200000)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--out", type=str, default="policy_params.npz")
    ap.add_argument("--resume", type=str, default=None,
                     help="path to an existing .npz checkpoint to continue from")
    args = ap.parse_args()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"device: {device}")

    torch.manual_seed(args.seed)
    rng = np.random.default_rng(args.seed)

    model = net_torch.PolicyValueNet().to(device)
    if args.resume:
        net_torch.load_npz_into(model, args.resume)
        print(f"resumed from {args.resume}")
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)

    replay_states, replay_policies, replay_values = [], [], []

    for it in range(1, args.iterations + 1):
        t0 = time.time()
        fn_batch = net_torch.make_batch_fn(model, device)

        data = self_play_batch(fn_batch, args.parallel, args.sims,
                                args.temp_moves, rng)
        wins = {1: 0, -1: 0, 2: 0}
        for states, policies, values, winner in data:
            replay_states.extend(states)
            replay_policies.extend(policies)
            replay_values.extend(values)
            wins[winner] += 1

        replay_states = replay_states[-args.max_replay:]
        replay_policies = replay_policies[-args.max_replay:]
        replay_values = replay_values[-args.max_replay:]

        X = np.stack(replay_states).astype(np.float32)
        P = np.stack(replay_policies).astype(np.float32)
        Z = np.array(replay_values, dtype=np.float32)

        avg_loss = train_step(model, optimizer, device, X, P, Z,
                               args.batch_size, args.epochs)

        dt = time.time() - t0
        print(f"iter {it:4d}/{args.iterations}  games={args.parallel}  "
              f"replay={X.shape[0]:6d}  loss={avg_loss[0]:.4f} "
              f"(pi={avg_loss[1]:.4f} v={avg_loss[2]:.4f})  "
              f"wins X={wins[1]} O={wins[-1]} draw={wins[2]}  [{dt:.1f}s]")

        net_torch.save_npz(model, args.out)

    print(f"saved trained params -> {args.out}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile train_kaggle_mp.py
"""Multi-process self-play + GPU training for Kaggle.

Why: the model is tiny, so a single batched NN forward call finishes almost
instantly -- the real bottleneck is the pure-Python MCTS tree traversal,
which is single-threaded (GIL) and only ever pins one CPU core. This script
fixes that by running self-play across `--workers` separate OS processes
(each pinning its own core, each doing CPU-only inference locally, no IPC
per NN call), while the main process does the actual training step on GPU
and re-saves weights each iteration for workers to pick up next round.

Usage:
    python train_kaggle_mp.py --iterations 150 --workers 4 \
        --parallel-per-worker 64 --sims 150 --out /kaggle/working/policy_params.npz
"""
import argparse
import multiprocessing as mp
import os
import time

import numpy as np
import torch

from game import UltimateTTT
import net_torch
from train_kaggle import self_play_batch, train_step


def _cpu_batch_fn(model):
    model.eval()

    @torch.no_grad()
    def fn_batch(states):
        x = torch.from_numpy(states.astype(np.float32))
        policy, value = model(x)
        return policy.numpy(), value.numpy()

    return fn_batch


def self_play_worker(task):
    """Runs entirely on CPU in a worker process: loads current weights from
    `ckpt_path`, plays `n_games` self-play games in lockstep, returns the
    training data. No CUDA/GPU touched here on purpose (see module docstring).
    """
    ckpt_path, n_games, sims, temp_moves, seed = task
    torch.set_num_threads(1)  # avoid oversubscription: each worker owns 1 core

    model = net_torch.PolicyValueNet()
    net_torch.load_npz_into(model, ckpt_path)
    fn_batch = _cpu_batch_fn(model)

    rng = np.random.default_rng(seed)
    return self_play_batch(fn_batch, n_games, sims, temp_moves, rng)


def _greedy_move(model, game):
    """No-search move: single forward pass, argmax over legal actions.
    Used only for the cheap periodic eval-vs-random check, not self-play."""
    x = torch.from_numpy(game.encode().astype(np.float32)).unsqueeze(0)
    with torch.no_grad():
        policy, _ = model(x)
    policy = policy.squeeze(0).cpu().numpy() * game.legal_action_mask()
    a = int(np.argmax(policy))
    return divmod(a, 9)


def _random_move(game, rng):
    moves = game.legal_moves()
    return moves[rng.integers(len(moves))]


def quick_eval_vs_random(model, n_games, rng):
    """Plays `model` (greedy) against random moves, alternating sides.
    Returns a score in [0, 1]: 1.0 = always wins, 0.5 = as good as random."""
    model.eval()
    wins = draws = losses = 0
    for i in range(n_games):
        trained_is_x = (i % 2 == 0)
        game = UltimateTTT()
        while not game.done:
            if (game.player == 1) == trained_is_x:
                sub, cell = _greedy_move(model, game)
            else:
                sub, cell = _random_move(game, rng)
            game.play(sub, cell)
        if game.winner == 2:
            draws += 1
        elif (game.winner == 1) == trained_is_x:
            wins += 1
        else:
            losses += 1
    return (wins + 0.5 * draws) / n_games, wins, losses, draws


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--iterations", type=int, default=150)
    ap.add_argument("--workers", type=int, default=None,
                     help="default: os.cpu_count()")
    ap.add_argument("--parallel-per-worker", type=int, default=64,
                     help="self-play games each worker runs simultaneously")
    ap.add_argument("--sims", type=int, default=150)
    ap.add_argument("--temp-moves", type=int, default=10)
    ap.add_argument("--batch-size", type=int, default=512)
    ap.add_argument("--epochs", type=int, default=2)
    ap.add_argument("--lr", type=float, default=1e-3)
    ap.add_argument("--max-replay", type=int, default=200000)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--out", type=str, default="policy_params.npz")
    ap.add_argument("--resume", type=str, default=None)
    ap.add_argument("--eval-every", type=int, default=10,
                     help="run a quick vs-random eval every N iterations (0 disables)")
    ap.add_argument("--eval-games", type=int, default=40)
    ap.add_argument("--patience", type=int, default=5,
                     help="stop if `eval-every`-spaced checks show no improvement "
                          "this many times in a row (0 disables early stopping)")
    ap.add_argument("--min-delta", type=float, default=0.01,
                     help="minimum score increase to count as an improvement")
    args = ap.parse_args()

    workers = args.workers or os.cpu_count() or 4
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"train device: {device}   self-play workers: {workers} (CPU, 1 core each)")
    print(f"games/iteration = {workers} workers x {args.parallel_per_worker} = "
          f"{workers * args.parallel_per_worker}")

    torch.manual_seed(args.seed)

    model = net_torch.PolicyValueNet().to(device)
    if args.resume:
        net_torch.load_npz_into(model, args.resume)
        print(f"resumed from {args.resume}")
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)

    # workers need a checkpoint file to load; write the initial (possibly
    # random) weights before the first round.
    net_torch.save_npz(model, args.out)

    replay_states, replay_policies, replay_values = [], [], []
    best_score = -1.0
    patience_counter = 0
    best_path = args.out.rsplit(".", 1)[0] + "_best.npz"
    eval_rng = np.random.default_rng(args.seed + 999)

    # IMPORTANT: force 'spawn'. The main process has already initialized a
    # CUDA context (model.to(device) above); on Linux the default start
    # method is 'fork', and forking a process that already holds a CUDA
    # context is unsafe -- workers can hang forever (looks exactly like
    # "stuck, no progress"). 'spawn' starts each worker as a fresh
    # interpreter, which is slower to launch but actually works.
    ctx = mp.get_context("spawn")
    with ctx.Pool(processes=workers) as pool:
        for it in range(1, args.iterations + 1):
            t0 = time.time()
            tasks = [
                (args.out, args.parallel_per_worker, args.sims, args.temp_moves,
                 args.seed * 100000 + it * 1000 + w)
                for w in range(workers)
            ]
            results = pool.map(self_play_worker, tasks)

            wins = {1: 0, -1: 0, 2: 0}
            for worker_data in results:
                for states, policies, values, winner in worker_data:
                    replay_states.extend(states)
                    replay_policies.extend(policies)
                    replay_values.extend(values)
                    wins[winner] += 1

            replay_states = replay_states[-args.max_replay:]
            replay_policies = replay_policies[-args.max_replay:]
            replay_values = replay_values[-args.max_replay:]

            X = np.stack(replay_states).astype(np.float32)
            P = np.stack(replay_policies).astype(np.float32)
            Z = np.array(replay_values, dtype=np.float32)

            avg_loss = train_step(model, optimizer, device, X, P, Z,
                                   args.batch_size, args.epochs)

            net_torch.save_npz(model, args.out)

            dt = time.time() - t0
            n_games = workers * args.parallel_per_worker
            print(f"iter {it:4d}/{args.iterations}  games={n_games}  "
                  f"replay={X.shape[0]:6d}  loss={avg_loss[0]:.4f} "
                  f"(pi={avg_loss[1]:.4f} v={avg_loss[2]:.4f})  "
                  f"wins X={wins[1]} O={wins[-1]} draw={wins[2]}  [{dt:.1f}s]")

            if args.eval_every > 0 and it % args.eval_every == 0:
                score, w, l, d = quick_eval_vs_random(model, args.eval_games, eval_rng)
                print(f"  eval vs random: score={score:.3f}  "
                      f"(win={w} loss={l} draw={d} / {args.eval_games})")

                if score > best_score + args.min_delta:
                    best_score = score
                    patience_counter = 0
                    net_torch.save_npz(model, best_path)
                    print(f"  new best ({score:.3f}) -> saved {best_path}")
                else:
                    patience_counter += 1
                    print(f"  no improvement over best ({best_score:.3f}): "
                          f"{patience_counter}/{args.patience}")
                    if args.patience > 0 and patience_counter >= args.patience:
                        print(f"early stopping at iteration {it}: "
                              f"no improvement for {args.patience} evals in a row")
                        break

    print(f"saved trained params -> {args.out}")
    if args.eval_every > 0:
        print(f"best checkpoint (score={best_score:.3f}) -> {best_path}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile export_weights.py
"""Export trained NumPy params (.npz) to a C header with float32 arrays.

Model is tiny (~93KB as float32), so we skip INT8 quantization entirely --
flash on the ESP32-S3 (8MB) has plenty of room, and it keeps the C inference
code (and this export script) dead simple.
"""
import argparse
import numpy as np


def c_array(name, arr):
    flat = arr.flatten().astype(np.float32)
    vals = ", ".join(f"{v:.8f}f" for v in flat)
    shape_comment = "x".join(str(d) for d in arr.shape)
    return (f"// shape: {shape_comment}\n"
            f"static const float {name}[{flat.size}] = {{{vals}}};\n")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("npz_path")
    ap.add_argument("--out", default="../main/model_weights.h")
    args = ap.parse_args()

    data = np.load(args.npz_path)
    params = {k: data[k] for k in data.files}

    lines = [
        "// Auto-generated by export_weights.py -- do not edit by hand.",
        "#pragma once",
        "",
    ]
    name_map = {
        "W1": "g_W1", "b1": "g_b1",
        "W2": "g_W2", "b2": "g_b2",
        "Wp": "g_Wp", "bp": "g_bp",
        "Wv": "g_Wv", "bv": "g_bv",
    }
    for key, cname in name_map.items():
        lines.append(c_array(cname, params[key]))

    lines.append(f"#define INPUT_DIM {params['W1'].shape[0]}")
    lines.append(f"#define H1_DIM {params['W1'].shape[1]}")
    lines.append(f"#define H2_DIM {params['W2'].shape[1]}")
    lines.append(f"#define POLICY_DIM {params['Wp'].shape[1]}")

    with open(args.out, "w") as f:
        f.write("\n".join(lines) + "\n")

    total_bytes = sum(params[k].size for k in name_map) * 4
    print(f"wrote {args.out}  ({total_bytes/1024:.1f} KB as float32)")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile eval_vs_random.py
"""Sanity check: trained policy (greedy, no search) vs a random-move player.

If training worked, the trained side should win the vast majority of games.
"""
import argparse
import numpy as np

from game import UltimateTTT
import net


def greedy_move(params, game):
    x = game.encode()
    policy, _ = net.predict_single(params, x)
    mask = game.legal_action_mask()
    policy = policy * mask
    a = int(np.argmax(policy))
    return divmod(a, 9)


def random_move(game, rng):
    moves = game.legal_moves()
    return moves[rng.integers(len(moves))]


def play_one(params, trained_is_x, rng):
    game = UltimateTTT()
    while not game.done:
        if (game.player == 1) == trained_is_x:
            sub, cell = greedy_move(params, game)
        else:
            sub, cell = random_move(game, rng)
        game.play(sub, cell)
    return game.winner


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("npz_path")
    ap.add_argument("--games", type=int, default=100)
    args = ap.parse_args()

    params = net.load_params(args.npz_path)
    rng = np.random.default_rng(42)

    trained_wins = 0
    random_wins = 0
    draws = 0
    for i in range(args.games):
        trained_is_x = (i % 2 == 0)
        winner = play_one(params, trained_is_x, rng)
        if winner == 2:
            draws += 1
        elif (winner == 1) == trained_is_x:
            trained_wins += 1
        else:
            random_wins += 1

    print(f"trained: {trained_wins}  random: {random_wins}  draws: {draws}  "
          f"(out of {args.games} games)")


if __name__ == "__main__":
    main()


## Train (multi-process self-play + GPU training + early stopping)

The model is tiny, so the MCTS tree traversal (pure Python, single-threaded) bottlenecks on one CPU core long before the GPU does any real work. `train_kaggle_mp.py` runs self-play across `--workers` separate processes (one per CPU core) while the main process trains on GPU -- this uses your Kaggle CPU quota, not just the GPU. Checkpoints save every iteration to `/kaggle/working/policy_params.npz`, so stopping early (or hitting the session limit) still leaves you a usable file.

Every `--eval-every` iterations it also plays a quick batch of games against a random-move opponent; if the score doesn't improve for `--patience` checks in a row, training stops automatically and the best checkpoint is saved separately as `policy_params_best.npz` -- so you don't have to guess `--iterations` up front or babysit the notebook.


In [ ]:
import os
print('CPU cores available:', os.cpu_count())


In [ ]:
!python train_kaggle_mp.py \
    --iterations 150 \
    --workers 4 \
    --parallel-per-worker 32 \
    --sims 80 \
    --eval-every 10 \
    --eval-games 40 \
    --patience 5 \
    --out /kaggle/working/policy_params.npz


## Evaluate against a random-move opponent

Use `policy_params_best.npz` (the early-stopping checkpoint), not the plain `policy_params.npz` -- it's whichever iteration scored highest, not necessarily the very last one.


In [ ]:
!python eval_vs_random.py /kaggle/working/policy_params_best.npz --games 200


## Export to a C header

Download `model_weights.h` afterwards (File panel on the right) and drop it into your ESP-IDF project's `main/` folder.


In [ ]:
!python export_weights.py /kaggle/working/policy_params_best.npz --out /kaggle/working/model_weights.h
